# ANN topological simplification pipeline

This notebook trains the requested feedforward models, captures hidden activations on a fixed probe set, runs ripser, computes COM, and plots COM vs architecture size.

Pipeline order:
1. Training
2. Activation capture
3. Ripser
4. COM
5. Plotting


In [ ]:
# Cell 1: setup

# !pip3 -q install ripser dill scipy seaborn pandas scikit-learn

import os
import json
import math
import random
from pathlib import Path

import dill
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from ripser import ripser
from scipy.stats import pearsonr, spearmanr, kendalltau

# Reproducibility
GLOBAL_SEED = 0
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# Paths
ROOT = Path('${TDL_ROOT_DIR}/John/ANN')
DATA_PATH = ROOT / 'full_dataset.npz'
MODEL_ROOT = ROOT / 'models'
RIPSER_ROOT = ROOT / 'ripser_results'
FIG_ROOT = ROOT / 'figures'
LOG_ROOT = ROOT / 'logs'
MODEL_SIZES_JSON = ROOT / 'model_sizes.json'

for p in [MODEL_ROOT, RIPSER_ROOT, FIG_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Experiment settings
N_SEEDS = 30
TARGET_ACC = 0.99
BATCH_SIZE = 256
MAX_EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 0.0
PATIENCE = 20

# Ripser / COM settings
ETA = 2.5
DIMS = (0,)
N_PERM = 15   # closest direct ripser analogue to the user's k = 15 request
USE_RUNNING_MIN = True
INCLUDE_OUTPUT = True
DIR_NAME = f'ripser_k{N_PERM}_eta{ETA}'


In [ ]:
# Cell 2: load data and build a fixed probe set

def load_npz_dataset(path: Path):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)

    # Common conventions
    if {'X_train', 'y_train', 'X_test', 'y_test'} <= keys:
        X_train = data['X_train']
        y_train = data['y_train']
        X_test = data['X_test']
        y_test = data['y_test']
        return (X_train, y_train), (X_test, y_test)

    if {'X', 'y'} <= keys:
        X = data['X']
        y = data['y']
        return (X, y), None

    # Fallback: try to infer the two largest arrays as X and y
    arrays = [(k, data[k]) for k in data.files if isinstance(data[k], np.ndarray)]
    if len(arrays) < 2:
        raise ValueError(f'Could not infer dataset arrays from {path}; found keys: {data.files}')

    arrays = sorted(arrays, key=lambda kv: kv[1].size, reverse=True)
    X = arrays[0][1]
    y = arrays[1][1]
    return (X, y), None

(dataset_train, dataset_test) = load_npz_dataset(DATA_PATH)
X_all, y_all = dataset_train

X_all = np.asarray(X_all)
y_all = np.asarray(y_all)

# Convert labels to integer class indices if needed
if y_all.ndim > 1:
    y_all = y_all.argmax(axis=-1)
y_all = y_all.astype(np.int64)

# Basic shape normalization
if X_all.ndim == 2:
    # [N, D] already fine
    pass
elif X_all.ndim == 3:
    # [N, C, L] or similar; flatten to vectors for a vanilla MLP
    X_all = X_all.reshape(X_all.shape[0], -1)
else:
    X_all = X_all.reshape(X_all.shape[0], -1)

print('X shape:', X_all.shape)
print('y shape:', y_all.shape)
print('num classes:', int(np.max(y_all)) + 1)

# Fixed probe set used for activation capture and ripser
# Keep this identical across every model.
PROBE_SIZE = min(1024, len(X_all))
probe_idx = np.random.RandomState(GLOBAL_SEED).choice(len(X_all), size=PROBE_SIZE, replace=False)
X_probe = X_all[probe_idx]
y_probe = y_all[probe_idx]

# Train/val split
N = len(X_all)
N_VAL = int(0.1 * N)
N_TRAIN = N - N_VAL
train_ds_full = TensorDataset(torch.tensor(X_all, dtype=torch.float32), torch.tensor(y_all, dtype=torch.long))
train_ds, val_ds = random_split(
    train_ds_full,
    [N_TRAIN, N_VAL],
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
probe_loader = DataLoader(TensorDataset(torch.tensor(X_probe, dtype=torch.float32)), batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_all.shape[1]
num_classes = int(np.max(y_all)) + 1
print('input_dim:', input_dim)
print('num_classes:', num_classes)
print('probe size:', PROBE_SIZE)


In [ ]:
# Cell 3: model grid

ARCHS = {
    '30x8': [30, 30, 30, 30, 30, 30, 30, 30],
    '24x8': [24, 24, 24, 24, 24, 24, 24, 24],
    '18x8': [18, 18, 18, 18, 18, 18, 18, 18],
    '30x4_24x4': [30, 30, 30, 30, 24, 24, 24, 24],
    '30x4_18x4': [30, 30, 30, 30, 18, 18, 18, 18],
    '30x4_12x4': [30, 30, 30, 30, 12, 12, 12, 12],
}

ACTIVATIONS = ['relu', 'tanh', 'leaky_relu']

MODEL_SIZES = {name: int(sum(dims)) for name, dims in ARCHS.items()}
with open(MODEL_SIZES_JSON, 'w') as f:
    json.dump(MODEL_SIZES, f, indent=2)

print(MODEL_SIZES)

# Save a small manifest for convenience
manifest = pd.DataFrame([
    {'arch': k, 'hidden_dims': v, 'hidden_sum': sum(v)} for k, v in ARCHS.items()
])
manifest


In [ ]:
# Cell 4: generic feedforward model and training utilities

class FeedForwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, activation_name='relu'):
        super().__init__()
        act_map = {
            'relu': nn.ReLU,
            'tanh': nn.Tanh,
            'leaky_relu': nn.LeakyReLU,
        }
        if activation_name not in act_map:
            raise ValueError(f'Unknown activation: {activation_name}')
        act_cls = act_map[activation_name]

        layers = []
        dims = [input_dim] + list(hidden_dims)
        for i in range(len(hidden_dims)):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(act_cls())
        layers.append(nn.Linear(dims[-1], num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_model(hidden_dims, activation_name):
    return FeedForwardNet(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        activation_name=activation_name,
    )


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=-1)
    return (preds == y).float().mean().item()


def evaluate(model, loader, device=DEVICE):
    model.eval()
    total_correct = 0
    total = 0
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            total_correct += (logits.argmax(dim=-1) == yb).sum().item()
            total += len(xb)
    return {'loss': total_loss / max(total, 1), 'acc': total_correct / max(total, 1)}


def train_one_model(hidden_dims, activation_name, seed, save_path, max_epochs=MAX_EPOCHS, target_acc=TARGET_ACC):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = build_model(hidden_dims, activation_name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = -1.0
    best_state = None
    best_epoch = -1
    patience_left = PATIENCE
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(xb)
            train_correct += (logits.argmax(dim=-1) == yb).sum().item()
            train_total += len(xb)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_metrics = evaluate(model, val_loader)

        row = {
            'seed': seed,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        }
        history.append(row)

        if val_metrics['acc'] > best_val_acc:
            best_val_acc = val_metrics['acc']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1

        if best_val_acc >= target_acc:
            break
        if patience_left <= 0:
            break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to('cpu')
    model.eval()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'state_dict': model.state_dict(),
        'hidden_dims': hidden_dims,
        'activation_name': activation_name,
        'seed': seed,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
    }, save_path)

    return model, pd.DataFrame(history)


In [ ]:
# Cell 5: train the full grid and save checkpoints

# Expected output layout:
# models/<arch>/<activation>/seed_<seed>.pth

train_summaries = []

for arch_name, hidden_dims in ARCHS.items():
    for act_name in ACTIVATIONS:
        for seed in range(N_SEEDS):
            ckpt_path = MODEL_ROOT / arch_name / act_name / f'seed_{seed}.pth'
            print(f'Training {arch_name} | {act_name} | seed {seed}')
            model, hist = train_one_model(hidden_dims, act_name, seed, ckpt_path)

            final_val = evaluate(model.to(DEVICE), val_loader)
            final_train = evaluate(model.to(DEVICE), train_loader)
            model.to('cpu')

            hist_path = LOG_ROOT / arch_name / act_name / f'seed_{seed}.csv'
            hist_path.parent.mkdir(parents=True, exist_ok=True)
            hist.to_csv(hist_path, index=False)

            train_summaries.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'train_acc': final_train['acc'],
                'val_acc': final_val['acc'],
                'train_loss': final_train['loss'],
                'val_loss': final_val['loss'],
            })

summary_df = pd.DataFrame(train_summaries)
summary_df.head()


In [ ]:
# Cell 6: activation capture helpers

def load_trained_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    hidden_dims = ckpt['hidden_dims']
    activation_name = ckpt['activation_name']
    model = build_model(hidden_dims, activation_name)
    model.load_state_dict(ckpt['state_dict'], strict=True)
    model.eval()
    return model, hidden_dims, activation_name


def get_hidden_linear_names(model):
    # All Linear layers except the final classifier layer
    names = []
    modules = list(model.named_modules())
    linear_names = [name for name, module in modules if isinstance(module, nn.Linear)]
    if len(linear_names) < 2:
        raise ValueError('Model must have at least one hidden Linear layer and one output Linear layer.')
    return linear_names[:-1]


@torch.no_grad()
def capture_hidden_activations(model, loader, layer_names):
    modules = dict(model.named_modules())
    for name in layer_names:
        if name not in modules:
            raise KeyError(f'Layer {name} not found in model. Available names include: {list(modules.keys())[:20]}')

    buffers = {name: [] for name in layer_names}
    handles = []

    def make_hook(layer_name):
        def hook(module, inp, out):
            y = out.detach().cpu()
            if y.ndim > 2:
                y = y.reshape(y.shape[0], -1)
            buffers[layer_name].append(y.numpy())
        return hook

    for name in layer_names:
        handles.append(modules[name].register_forward_hook(make_hook(name)))

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _ = model.to(DEVICE)(xb)

    for h in handles:
        h.remove()

    activations = {name: np.concatenate(buffers[name], axis=0) for name in layer_names}
    return activations


In [ ]:
# Cell 7: capture activations for every checkpoint

ACT_ROOT = ROOT / 'activations'
ACT_ROOT.mkdir(parents=True, exist_ok=True)

activation_manifest = []

for arch_name in ARCHS:
    for act_name in ACTIVATIONS:
        ckpt_dir = MODEL_ROOT / arch_name / act_name
        if not ckpt_dir.exists():
            print('Skipping missing', ckpt_dir)
            continue

        for seed in range(N_SEEDS):
            ckpt_path = ckpt_dir / f'seed_{seed}.pth'
            if not ckpt_path.exists():
                print('Missing checkpoint:', ckpt_path)
                continue

            model, hidden_dims, activation_name = load_trained_model(ckpt_path)
            layer_names = get_hidden_linear_names(model)
            acts = capture_hidden_activations(model, probe_loader, layer_names)

            out_dir = ACT_ROOT / arch_name / act_name / f'seed_{seed}'
            out_dir.mkdir(parents=True, exist_ok=True)

            # Save the fixed probe input as the baseline 'input layer' point cloud
            with open(out_dir / 'input_layer.pkl', 'wb') as f:
                dill.dump(X_probe, f)

            for lname, A in acts.items():
                safe_name = lname.replace('.', '_')
                with open(out_dir / f'{safe_name}.pkl', 'wb') as f:
                    dill.dump(A, f)

            activation_manifest.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'activation_dir': str(out_dir),
                'layer_names': layer_names,
            })

activation_manifest_df = pd.DataFrame(activation_manifest)
activation_manifest_df.head()


In [ ]:
# Cell 8: ripser utilities

def standardize_features(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mu = x.mean(axis=0, keepdims=True)
    sd = x.std(axis=0, keepdims=True)
    return (x - mu) / (sd + eps)


def compute_diagrams_for_point_cloud(X, maxdim=1, standardize=True, n_perm=15):
    X = np.asarray(X, dtype=np.float32)
    ok = np.isfinite(X).all(axis=1)
    X = X[ok]
    if X.shape[0] < 3:
        raise ValueError('Need at least 3 points for ripser.')
    if standardize:
        X = standardize_features(X)
    return ripser(X, maxdim=maxdim, n_perm=n_perm)['dgms']


def save_diagrams(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        dill.dump(obj, f)


def load_pickle(path):
    with open(path, 'rb') as f:
        return dill.load(f)


In [ ]:
# Cell 9: run ripser on every captured activation set

RIP_ROOT = RIPSER_ROOT / DIR_NAME
RIP_ROOT.mkdir(parents=True, exist_ok=True)

ripser_manifest = []

for row in activation_manifest:
    arch_name = row['arch']
    act_name = row['activation']
    seed = row['seed']
    act_dir = Path(row['activation_dir'])

    out_dir = RIP_ROOT / arch_name / act_name / f'seed_{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)

    input_cloud = load_pickle(act_dir / 'input_layer.pkl')
    input_dgm = compute_diagrams_for_point_cloud(
        input_cloud,
        maxdim=1,
        standardize=True,
        n_perm=N_PERM,
    )
    save_diagrams(input_dgm, out_dir / 'input_layer.pkl')

    layer_files = sorted([p for p in act_dir.glob('*.pkl') if p.name != 'input_layer.pkl'])
    layer_diagrams = []

    for p in layer_files:
        A = load_pickle(p)
        dgm = compute_diagrams_for_point_cloud(
            A,
            maxdim=1,
            standardize=True,
            n_perm=N_PERM,
        )
        layer_diagrams.append(dgm)

    save_diagrams(layer_diagrams, out_dir / 'model.pkl')

    ripser_manifest.append({
        'arch': arch_name,
        'activation': act_name,
        'seed': seed,
        'ripser_dir': str(out_dir),
        'num_layers': len(layer_diagrams),
    })

ripser_manifest_df = pd.DataFrame(ripser_manifest)
ripser_manifest_df.head()


In [ ]:
# Cell 10: COM utilities

def betti_at_eta_one_dim(diagram_one_dim, eta):
    if diagram_one_dim is None:
        return 0.0

    arr = np.asarray(diagram_one_dim)

    if arr.size == 0:
        return 0.0

    # 🔑 FIX: ensure shape is (N, 2)
    if arr.ndim == 1:
        if arr.shape[0] == 2:
            arr = arr.reshape(1, 2)
        else:
            return 0.0

    births = arr[:, 0]
    deaths = arr[:, 1]

    alive = (births <= eta) & (eta < deaths)
    return float(np.sum(alive))


def betti_at_eta(diagram, eta, dim=0):
    if diagram is None:
        return 0.0
    if dim >= len(diagram):
        return 0.0
    return betti_at_eta_one_dim(diagram[dim], eta)


def com_of_drops_one_seed(curve, use_running_min=True, include_output=True, no_drop_value=None):
    curve = np.asarray(curve, dtype=float).copy()
    if use_running_min:
        curve = np.minimum.accumulate(curve)
    if not include_output and len(curve) > 1:
        curve = curve[:-1]
    if len(curve) < 2:
        return np.nan if no_drop_value is None else no_drop_value
    drops = curve[:-1] - curve[1:]
    total_drop = drops.sum()
    if total_drop <= 0:
        return np.nan if no_drop_value is None else no_drop_value
    layers = np.arange(1, len(curve))
    return float(np.sum(layers * drops) / total_drop)


def get_betti_mat_from_saved_ripser(ripser_dir, eta=ETA, dims=DIMS, use_running_min=False):
    ripser_dir = Path(ripser_dir)
    layer_path = ripser_dir / 'model.pkl'
    input_path = ripser_dir / 'input_layer.pkl'

    all_diagrams = load_pickle(layer_path)
    input_layer_dgm = load_pickle(input_path) if input_path.exists() else None

    K = len(all_diagrams)
    L = len(all_diagrams[0])
    betti_mat = np.zeros((K, L), dtype=float)

    for j, diagrams in enumerate(all_diagrams):
        for ell, diagram in enumerate(diagrams):
            entry = 0.0
            for dim in dims:
                entry += betti_at_eta(diagram, eta=eta, dim=dim)
            betti_mat[j, ell] = entry

    input_b = 0.0
    if input_layer_dgm is not None:
        for dim in dims:
            input_b += betti_at_eta(input_layer_dgm, eta=eta, dim=dim)

    betti_mat = np.hstack([np.full((K, 1), input_b), betti_mat])

    if use_running_min:
        for i in range(len(betti_mat)):
            betti_mat[i] = np.minimum.accumulate(betti_mat[i])

    return betti_mat


def get_com_from_saved_ripser(ripser_dir, eta=ETA, dims=DIMS, use_running_min=False, include_output=INCLUDE_OUTPUT):
    betti_mat = get_betti_mat_from_saved_ripser(
        ripser_dir=ripser_dir,
        eta=eta,
        dims=dims,
        use_running_min=False,
    )

    K, n_layers_total = betti_mat.shape
    no_drop_value = float(n_layers_total)
    com = np.empty(K, dtype=float)

    for i in range(K):
        com[i] = com_of_drops_one_seed(
            betti_mat[i],
            use_running_min=use_running_min,
            include_output=include_output,
            no_drop_value=no_drop_value,
        )
    return com


In [ ]:
# Cell 11: compute COM for every model

com_rows = []

for row in ripser_manifest:
    arch_name = row['arch']
    act_name = row['activation']
    seed = row['seed']
    ripser_dir = Path(row['ripser_dir'])

    com = get_com_from_saved_ripser(
        ripser_dir=ripser_dir,
        eta=ETA,
        dims=DIMS,
        use_running_min=USE_RUNNING_MIN,
        include_output=INCLUDE_OUTPUT,
    )

    for i, val in enumerate(com):
        com_rows.append({
            'arch': arch_name,
            'activation': act_name,
            'seed': seed,
            'model_idx': i,
            'COM': float(val),
            'hidden_sum': MODEL_SIZES[arch_name],
        })

com_df = pd.DataFrame(com_rows)
com_df.head()


In [ ]:
# Cell 12: aggregate statistics and correlations

agg = com_df.groupby(['arch', 'activation'], as_index=False).agg(
    mean_com=('COM', 'mean'),
    std_com=('COM', 'std'),
    n=('COM', 'size'),
    hidden_sum=('hidden_sum', 'first'),
)

agg = agg.sort_values(['hidden_sum', 'arch', 'activation'])
agg


In [ ]:
# Cell 13: plot COM results

sns.set_style('whitegrid')
sns.set_context('talk')

plt.figure(figsize=(11, 7))

# Scatter all seeds, color by activation, x by hidden sum
palette = {'relu': 'tab:blue', 'tanh': 'tab:orange', 'leaky_relu': 'tab:green'}
marker_map = {'relu': 'o', 'tanh': 's', 'leaky_relu': '^'}

for act in ACTIVATIONS:
    subset = com_df[com_df['activation'] == act]
    plt.scatter(
        subset['hidden_sum'],
        subset['COM'],
        s=45,
        alpha=0.55,
        color=palette[act],
        marker=marker_map[act],
        edgecolor='black',
        linewidth=0.3,
        label=act,
    )

# Overlay mean per architecture/activation
for act in ACTIVATIONS:
    subset = agg[agg['activation'] == act].sort_values('hidden_sum')
    plt.plot(
        subset['hidden_sum'],
        subset['mean_com'],
        color=palette[act],
        linewidth=2,
        alpha=0.85,
    )

plt.xlabel('Sum of Hidden Layer Sizes', fontsize=14)
plt.ylabel('COM', fontsize=14)
plt.title('COM vs. model size for ANN architectures', fontsize=16)
plt.legend(title='Activation', frameon=True)
plt.tight_layout()

out_fig = FIG_ROOT / 'com_vs_model_size_ann.png'
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()

print('saved figure:', out_fig)


In [ ]:
print(com_df.columns)

In [ ]:
# Cell 13: clean 3-panel COM vs hidden-size plot

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

df_plot = com_df.copy()

# Make sure the column names match
if "COM" not in df_plot.columns and "com" in df_plot.columns:
    df_plot = df_plot.rename(columns={"com": "COM"})
if "arch" not in df_plot.columns and "architecture" in df_plot.columns:
    df_plot = df_plot.rename(columns={"architecture": "arch"})

ARCH_ORDER = ["30x8", "24x8", "18x8", "30x4_24x4", "30x4_18x4", "30x4_12x4"]
ACT_ORDER = ["relu", "tanh", "leaky_relu"]
ACT_LABELS = {
    "relu": "ReLU Activations",
    "tanh": "Tanh Activations",
    "leaky_relu": "Leaky ReLU Activations",
}

df_plot["arch"] = pd.Categorical(df_plot["arch"], categories=ARCH_ORDER, ordered=True)
df_plot["activation"] = pd.Categorical(df_plot["activation"], categories=ACT_ORDER, ordered=True)

sns.set_style("whitegrid")
sns.set_context("talk")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, act in zip(axes, ACT_ORDER):
    sub = df_plot[df_plot["activation"] == act].copy()
    sub = sub.sort_values("hidden_sum")

    x = sub["hidden_sum"].astype(float).to_numpy()
    y = sub["COM"].astype(float).to_numpy()

    # Seed-level points
    ax.scatter(
        x,
        y,
        s=50,
        marker="s",
        alpha=0.9,
        edgecolor="black",
        linewidth=0.5,
    )

    # LOWESS trend
    if len(sub) >= 3:
        sns.regplot(
            x=x,
            y=y,
            scatter=False,
            ax=ax,
            color="dimgray",
            line_kws={"lw": 1.6, "ls": "--", "alpha": 0.9},
        )

    # Architecture mean points and labels
    mean_pts = sub.groupby("arch", as_index=False).agg(
        hidden_sum=("hidden_sum", "first"),
        COM=("COM", "mean"),
    )

    ax.scatter(
        mean_pts["hidden_sum"],
        mean_pts["COM"],
        s=110,
        marker="D",
        facecolor="white",
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
    )

    for _, row in mean_pts.iterrows():
        ax.annotate(
            str(row["arch"]),
            (row["hidden_sum"], row["COM"]),
            textcoords="offset points",
            xytext=(4, 4),
            fontsize=9,
        )

    # Correlations
    if len(sub) >= 2:
        spear_r, spear_p = spearmanr(x, y)
        corr_txt = rf"$\rho_s = {spear_r:.2f}$" + f"\n(p={spear_p:.3g})"
        ax.text(
            0.03,
            0.03,
            corr_txt,
            transform=ax.transAxes,
            fontsize=11,
            va="bottom",
            ha="left",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.75, edgecolor="gray"),
        )

    ax.set_title(ACT_LABELS[act], fontsize=14, pad=8)
    ax.set_xlabel("Sum of Hidden Layer Sizes", fontsize=12)
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel("COM", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: plot all individual Betti curves for each architecture

from pathlib import Path
import numpy as np
import dill
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
ETA = 2.5
DIMS_TO_PLOT = (0,)   # change to (0, 1) if you want beta0 + beta1 summed
USE_RUNNING_MIN = False

ARCH_ORDER = ["30x8", "24x8", "18x8", "30x4_24x4", "30x4_18x4", "30x4_12x4"]
ACT_ORDER = ["relu", "tanh", "leaky_relu"]

ACT_COLORS = {
    "relu": "tab:blue",
    "tanh": "tab:orange",
    "leaky_relu": "tab:green",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def betti_at_eta_one_dim(diagram_one_dim, eta):
    arr = np.asarray(diagram_one_dim)

    if arr.size == 0:
        return 0.0

    # handle single point case: shape (2,)
    if arr.ndim == 1:
        if arr.shape[0] == 2:
            arr = arr.reshape(1, 2)
        else:
            return 0.0

    births = arr[:, 0]
    deaths = arr[:, 1]
    alive = (births <= eta) & (eta < deaths)
    return float(np.sum(alive))


def betti_at_eta(diagram, eta, dim=0):
    if diagram is None or dim >= len(diagram):
        return 0.0
    return betti_at_eta_one_dim(diagram[dim], eta)


def curve_from_diagram_list(diagrams, eta=ETA, dims=DIMS_TO_PLOT, use_running_min=USE_RUNNING_MIN):
    """
    diagrams: list of layer diagrams, one per layer.
    Returns a 1D curve across layers.
    """
    curve = []
    for layer_dgm in diagrams:
        val = 0.0
        for dim in dims:
            val += betti_at_eta(layer_dgm, eta=eta, dim=dim)
        curve.append(val)

    curve = np.asarray(curve, dtype=float)

    if use_running_min:
        curve = np.minimum.accumulate(curve)

    return curve


def load_diagram(path):
    with open(path, "rb") as f:
        return dill.load(f)

RIPSER_ROOT = Path("ripser_results") / f"ripser_k15_eta{ETA}"
def get_seed_ripser_dir(arch_name, activation, seed):
    """
    Adjust this if your folder structure is different.
    Expected:
        RIPSER_ROOT/<arch>/<activation>/seed_<seed>/
    """
    return Path(RIPSER_ROOT) / arch_name / activation / f"seed_{seed}"


def collect_curves_for_architecture(arch_name):
    """
    Returns:
        curves_by_act[activation] = list of 1D numpy arrays
    """
    curves_by_act = {act: [] for act in ACT_ORDER}

    for act in ACT_ORDER:
        for seed in range(30):
            ripser_dir = get_seed_ripser_dir(arch_name, act, seed)
            if not ripser_dir.exists():
                continue

            input_path = ripser_dir / "input_layer.pkl"

            # all model_* pkl files, or any pkl except input_layer.pkl
            model_files = sorted([
                p for p in ripser_dir.glob("*.pkl")
                if p.name != "input_layer.pkl"
            ])

            if len(model_files) == 0:
                continue

            input_curve = None
            if input_path.exists():
                input_dgm = load_diagram(input_path)
                # input layer is usually a diagram list too
                input_curve = curve_from_diagram_list([input_dgm], eta=ETA, dims=DIMS_TO_PLOT, use_running_min=USE_RUNNING_MIN)[0]

            for mf in model_files:
                diagrams = load_diagram(mf)
                curve = curve_from_diagram_list(diagrams, eta=ETA, dims=DIMS_TO_PLOT, use_running_min=USE_RUNNING_MIN)

                if input_curve is not None:
                    curve = np.concatenate([[input_curve], curve])

                curves_by_act[act].append(curve)

    return curves_by_act


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
sns.set_style("whitegrid")
sns.set_context("talk")

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for ax, arch_name in zip(axes, ARCH_ORDER):
    curves_by_act = collect_curves_for_architecture(arch_name)

    max_len = 0
    for act in ACT_ORDER:
        if len(curves_by_act[act]) == 0:
            continue
        max_len = max(max_len, max(len(c) for c in curves_by_act[act]))

    if max_len == 0:
        ax.set_title(f"{arch_name} (no curves found)")
        ax.axis("off")
        continue

    # Plot all individual curves in light alpha
    for act in ACT_ORDER:
        curves = curves_by_act[act]
        if len(curves) == 0:
            continue

        for curve in curves:
            x = np.arange(1,len(curve)+1)
            ax.plot(
                x,
                curve,
                color=ACT_COLORS[act],
                alpha=0.12,
                lw=0.8,
            )

        # Mean curve on top
        min_len = min(len(c) for c in curves)
        stacked = np.vstack([c[:min_len] for c in curves])
        mean_curve = stacked.mean(axis=0)

        ax.plot(
            np.arange(1,min_len+1),
            mean_curve,
            color=ACT_COLORS[act],
            lw=2.4,
            label=act if arch_name == ARCH_ORDER[0] else None,
        )

    ax.set_title(arch_name, fontsize=13)
    ax.set_xlabel("Layer", fontsize=11)
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel(r"Betti count ($\beta$)", fontsize=11)
axes[3].set_ylabel(r"Betti count ($\beta$)", fontsize=11)

# Figure legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    [lab.replace("_", " ").title() for lab in ACT_ORDER],
    loc="upper center",
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 1.02),
    title="Activation",
)

title_dims = "+".join([f"β{d}" for d in DIMS_TO_PLOT])
fig.suptitle(f"Individual Betti Curves by Architecture  ({title_dims}, η={ETA})", y=1.06, fontsize=16)
plt.tight_layout()
plt.show()

## Optional: pairwise summary by architecture

This cell is useful if you want a paper-style table of mean COM by architecture and activation.


In [ ]:
# Optional summary table
summary_table = agg.pivot(index='arch', columns='activation', values='mean_com')
summary_table


In [ ]:
ETA_LIST = [1.5, 2.5, 3.5]
DIMS_TO_PLOT = (0,)
USE_RUNNING_MIN = False

In [ ]:
import numpy as np
import dill
from pathlib import Path

def betti_at_eta_one_dim(diagram_one_dim, eta):
    if diagram_one_dim is None:
        return 0.0

    arr = np.asarray(diagram_one_dim)

    if arr.size == 0:
        return 0.0

    # Handle the single-point case: shape (2,)
    if arr.ndim == 1:
        if arr.shape[0] == 2:
            arr = arr.reshape(1, 2)
        else:
            return 0.0

    births = arr[:, 0]
    deaths = arr[:, 1]
    alive = (births <= eta) & (eta < deaths)
    return float(np.sum(alive))


def betti_at_eta(diagram, eta, dim=0):
    if diagram is None or dim >= len(diagram):
        return 0.0
    return betti_at_eta_one_dim(diagram[dim], eta)


def curve_from_diagram_list(diagrams, eta, dims=(0,), use_running_min=False):
    curve = []
    for layer_dgm in diagrams:
        val = 0.0
        for dim in dims:
            val += betti_at_eta(layer_dgm, eta=eta, dim=dim)
        curve.append(val)

    curve = np.asarray(curve, dtype=float)

    if use_running_min:
        curve = np.minimum.accumulate(curve)

    return curve


def load_diagram(path):
    with open(path, "rb") as f:
        return dill.load(f)

In [ ]:
# Cell: build index_df from your ripser_results folder

from pathlib import Path
import pandas as pd

rows = []

for arch in ARCH_ORDER:
    for act in ACT_ORDER:
        base_dir = Path(RIPSER_ROOT) / arch / act

        if not base_dir.exists():
            continue

        for seed_dir in sorted(base_dir.glob("seed_*")):
            if not seed_dir.is_dir():
                continue

            # extract seed number
            seed_str = seed_dir.name.replace("seed_", "")
            try:
                seed = int(seed_str)
            except:
                continue

            rows.append({
                "arch": arch,
                "activation": act,
                "seed": seed,
                "model_idx": 0,  # you can keep this as 0 (not used)
                "ripser_dir": str(seed_dir),
            })

index_df = pd.DataFrame(rows)

print(f"Found {len(index_df)} runs")
index_df.head()

In [ ]:
def get_com_from_saved_ripser(ripser_dir, eta, dims=(0,), use_running_min=False, include_output=True):
    """
    ripser_dir should contain:
        input_layer.pkl
        model_*.pkl
    """
    ripser_dir = Path(ripser_dir)

    input_path = ripser_dir / "input_layer.pkl"
    model_files = sorted([p for p in ripser_dir.glob("*.pkl") if p.name != "input_layer.pkl"])

    if len(model_files) == 0:
        raise FileNotFoundError(f"No model_*.pkl files found in {ripser_dir}")

    input_b = 0.0
    if input_path.exists():
        input_dgm = load_diagram(input_path)
        for dim in dims:
            input_b += betti_at_eta(input_dgm, eta=eta, dim=dim)

    betti_mat = []

    for mf in model_files:
        diagrams = load_diagram(mf)

        curve = []
        for layer_dgm in diagrams:
            val = 0.0
            for dim in dims:
                val += betti_at_eta(layer_dgm, eta=eta, dim=dim)
            curve.append(val)

        curve = np.asarray(curve, dtype=float)

        # prepend input layer
        # curve = np.concatenate([[input_b], curve])

        if use_running_min:
            curve = np.minimum.accumulate(curve)

        betti_mat.append(curve)

    betti_mat = np.asarray(betti_mat, dtype=float)

    no_drop_value = float(betti_mat.shape[1])

    com = np.empty(betti_mat.shape[0], dtype=float)
    for i in range(betti_mat.shape[0]):
        com[i] = com_of_drops_one_seed(
            betti_mat[i],
            use_running_min=use_running_min,
            include_output=include_output,
            no_drop_value=no_drop_value,
        )

    return com


# Build a dataframe with COM at each eta
com_rows = []

for _, row in index_df.iterrows():   # assumes your saved-ripser index dataframe is named index_df
    arch_name = row["arch"]
    act_name = row["activation"]
    seed = row["seed"]
    model_idx = row["model_idx"]
    ripser_dir = Path(row["ripser_dir"])

    for eta in ETA_LIST:
        com = get_com_from_saved_ripser(
            ripser_dir=ripser_dir,
            eta=eta,
            dims=DIMS_TO_PLOT,
            use_running_min=USE_RUNNING_MIN,
            include_output=True,
        )

        for i, val in enumerate(com):
            com_rows.append({
                "arch": arch_name,
                "activation": act_name,
                "seed": seed,
                "model_idx": model_idx,
                "eta": eta,
                "COM": float(val),
                "hidden_sum": MODEL_SIZES[arch_name],
            })

com_eta_df = pd.DataFrame(com_rows)
com_eta_df.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ARCH_ORDER = ["30x8", "24x8", "18x8", "30x4_24x4", "30x4_18x4", "30x4_12x4"]
ACT_ORDER = ["relu", "tanh", "leaky_relu"]
ETA_ORDER = sorted(ETA_LIST)

ACT_LABELS = {
    "relu": "ReLU",
    "tanh": "Tanh",
    "leaky_relu": "Leaky ReLU",
}

sns.set_style("whitegrid")
sns.set_context("talk")

fig, axes = plt.subplots(1, len(ETA_ORDER), figsize=(6 * len(ETA_ORDER), 5), sharey=True)

if len(ETA_ORDER) == 1:
    axes = [axes]

for ax, eta in zip(axes, ETA_ORDER):
    sub_eta = com_eta_df[com_eta_df["eta"] == eta].copy()

    for act in ACT_ORDER:
        sub = sub_eta[sub_eta["activation"] == act].copy()
        if sub.empty:
            continue

        # architecture means
        mean_pts = sub.groupby("arch", as_index=False).agg(
            hidden_sum=("hidden_sum", "first"),
            COM=("COM", "mean"),
        ).sort_values("hidden_sum")

        x = mean_pts["hidden_sum"].astype(float).to_numpy()
        y = mean_pts["COM"].astype(float).to_numpy()

        ax.scatter(
            x,
            y,
            s=60,
            alpha=0.9,
            edgecolor="black",
            linewidth=0.5,
            label=ACT_LABELS[act],
        )

        if len(mean_pts) >= 3:
            sns.regplot(
                x=x,
                y=y,
                scatter=False,
                ax=ax,
                color="dimgray",
                line_kws={"lw": 1.5, "ls": "--", "alpha": 0.8},
            )

        if len(x) >= 2:
            rho, p = spearmanr(x, y)
            ax.text(
                0.03,
                0.03 + 0.08 * ACT_ORDER.index(act),
                rf"{ACT_LABELS[act]}: $\rho_s={rho:.2f}$",
                transform=ax.transAxes,
                fontsize=10,
                va="bottom",
                ha="left",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6, edgecolor="gray"),
            )

    ax.set_title(rf"$\eta={eta}$", fontsize=14)
    ax.set_xlabel("Sum of Hidden Layer Sizes", fontsize=12)
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel("COM", fontsize=12)
axes[-1].legend(title="Activation", frameon=True, loc="upper left")
plt.tight_layout()
plt.show()

filename = FIG_ROOT / f"com_vs_sum_of_hidden_layers_eta{eta}_B{''.join(map(str, DIMS_TO_PLOT))}.png"

plt.savefig(
    filename,
    dpi=300,
    bbox_inches="tight"
)

print(f"Saved: {filename}")

plt.show()

In [ ]:
# Cell: plot Betti curves for all eta values + SAVE, without input layer

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill

FIG_ROOT = Path("${TDL_ROOT_DIR}/John/ANN/figures/betti_curves")
FIG_ROOT.mkdir(parents=True, exist_ok=True)

ACT_COLORS = {
    "relu": "tab:blue",
    "tanh": "tab:orange",
    "leaky_relu": "tab:green",
}

def collect_curves_for_architecture_eta(arch_name, eta):
    """
    Returns:
        curves_by_act[activation] = list of 1D arrays
    """
    curves_by_act = {act: [] for act in ACT_ORDER}

    for act in ACT_ORDER:
        sub = index_df[
            (index_df["arch"] == arch_name) &
            (index_df["activation"] == act)
        ]

        for _, row in sub.iterrows():
            ripser_dir = Path(row["ripser_dir"])
            model_files = sorted([p for p in ripser_dir.glob("*.pkl") if p.name != "input_layer.pkl"])

            if len(model_files) == 0:
                continue

            for mf in model_files:
                diagrams = load_diagram(mf)

                curve = []
                for layer_dgm in diagrams:
                    val = 0.0
                    for dim in DIMS_TO_PLOT:
                        val += betti_at_eta(layer_dgm, eta=eta, dim=dim)
                    curve.append(val)

                curve = np.asarray(curve, dtype=float)

                if USE_RUNNING_MIN:
                    curve = np.minimum.accumulate(curve)

                curves_by_act[act].append(curve)

    return curves_by_act


for eta in ETA_LIST:
    sns.set_style("whitegrid")
    sns.set_context("talk")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
    axes = axes.flatten()

    for ax, arch_name in zip(axes, ARCH_ORDER):
        curves_by_act = collect_curves_for_architecture_eta(arch_name, eta)

        found_any = False

        for act in ACT_ORDER:
            curves = curves_by_act[act]
            if len(curves) == 0:
                continue

            found_any = True

            # individual curves
            for curve in curves:
                ax.plot(
                    np.arange(1,len(curve)+1),
                    curve,
                    color=ACT_COLORS[act],
                    alpha=0.10,
                    lw=0.8,
                )

            # mean curve
            min_len = min(len(c) for c in curves)
            stacked = np.vstack([c[:min_len] for c in curves])
            mean_curve = stacked.mean(axis=0)

            ax.plot(
                np.arange(1,min_len+1),
                mean_curve,
                color=ACT_COLORS[act],
                lw=2.2,
                label=act if arch_name == ARCH_ORDER[0] else None,
            )

        if not found_any:
            ax.set_title(f"{arch_name} (no curves)")
            ax.axis("off")
            continue

        ax.set_title(arch_name, fontsize=13)
        ax.set_xlabel("Hidden Layer", fontsize=11)
        ax.grid(True, alpha=0.2)

    axes[0].set_ylabel(r"Betti count ($\beta$)", fontsize=11)
    axes[3].set_ylabel(r"Betti count ($\beta$)", fontsize=11)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        [lab.replace("_", " ").title() for lab in ACT_ORDER],
        loc="upper center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, 1.02),
        title="Activation",
    )

    title_dims = "+".join([f"β{d}" for d in DIMS_TO_PLOT])
    fig.suptitle(f"Betti Curves by Architecture, η={eta}  ({title_dims})", y=1.05, fontsize=16)

    plt.tight_layout()

    filename = FIG_ROOT / f"betti_curves_no_input_eta{eta}_B{''.join(map(str, DIMS_TO_PLOT))}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Saved: {filename}")

    plt.show()

In [ ]:
# Debug COM values: check whether they are actually integers

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
np.set_printoptions(precision=12, suppress=True)

# 1) Basic summary
print("dtype:", com_df["COM"].dtype)
print(com_df["COM"].describe())

# 2) Show unique values with high precision
vals = np.sort(com_df["COM"].to_numpy())
print("\nFirst 50 COM values with 12 decimals:")
for v in vals[:50]:
    print(f"{v:.12f}")

print("\nNumber of unique COM values:", com_df["COM"].nunique())

# 3) Check how many are essentially integers
is_int = np.isclose(com_df["COM"], np.round(com_df["COM"]), atol=1e-10)
print("\nCounts of near-integer values:")
print(is_int.value_counts())

# 4) Show rows that are not near integers
non_int = com_df.loc[~is_int, ["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]]
print("\nNon-integer COM rows:")
print(non_int.head(100).to_string(index=False))

# 5) Show rows near 1 or 2 specifically
near_1 = com_df[np.isclose(com_df["COM"], 1.0, atol=1e-10)]
near_2 = com_df[np.isclose(com_df["COM"], 2.0, atol=1e-10)]

print(f"\nRows with COM very close to 1.0: {len(near_1)}")
print(near_1[["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]].head(20).to_string(index=False))

print(f"\nRows with COM very close to 2.0: {len(near_2)}")
print(near_2[["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]].head(20).to_string(index=False))

In [ ]:
print(com_df["COM"].value_counts(dropna=False).sort_index())

vals = np.sort(com_df["COM"].dropna().unique())
for v in vals[:50]:
    print(f"{v:.15f}")

check = com_df.loc[
    ~np.isclose(com_df["COM"], np.round(com_df["COM"]), atol=1e-10),
    ["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]
]
print(check.head(50).to_string(index=False))

In [ ]:
print(com_df["COM"].value_counts(dropna=False).sort_index())

vals = np.sort(com_df["COM"].dropna().unique())
for v in vals[:50]:
    print(f"{v:.15f}")

check = com_df.loc[
    ~np.isclose(com_df["COM"], np.round(com_df["COM"]), atol=1e-10),
    ["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]
]
print(check.head(50).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np

ripser_dir = Path(index_df.iloc[5]["ripser_dir"])
eta = 2.5
dims = (0,)

model_files = sorted([p for p in ripser_dir.glob("*.pkl") if p.name != "input_layer.pkl"])
mf = model_files[0]
diagrams = load_diagram(mf)

curve = []
for layer_idx, layer_dgm in enumerate(diagrams, start=1):
    val = 0.0
    for dim in dims:
        val += betti_at_eta(layer_dgm, eta=eta, dim=dim)
    curve.append(val)

curve = np.asarray(curve, dtype=float)
drops = np.maximum(curve[:-1] - curve[1:], 0.0)
layers = np.arange(1, len(curve))

print("file:", mf.name)
print("num layers:", len(curve))
print("curve by layer:")
for i, v in enumerate(curve, start=1):
    print(f"  layer {i}: {v}")

print("drops between layers:")
for i, d in enumerate(drops, start=1):
    print(f"  drop {i}->{i+1}: {d}")

total_drop = drops.sum()
if total_drop > 0:
    com = np.sum(layers * drops) / total_drop
else:
    com = np.nan

print("COM:", com)

In [ ]:
print(com_df[["arch", "activation", "seed", "model_idx", "COM"]].head(20).to_string(index=False))
print(com_df["COM"].value_counts().sort_index().head(20))

In [ ]:
# Check exact COM values, not rounded display
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

print("COM dtype:", com_df["COM"].dtype)

u = np.sort(com_df["COM"].dropna().unique())
print("\nUnique COM values with full precision:")
for v in u:
    print(f"{v:.15f}")

print("\nCounts of each exact COM value:")
print(com_df["COM"].value_counts(dropna=False).sort_index())

print("\nRows with non-integer COM values:")
non_int = com_df[~np.isclose(com_df["COM"], np.round(com_df["COM"]), atol=1e-10)]
print(non_int[["arch", "activation", "seed", "model_idx", "COM", "hidden_sum"]].head(100).to_string(index=False))

print("\nCheck whether a rounded or grouped dataframe was used:")
if "agg" in globals():
    print("agg columns:", agg.columns)
    print(agg.head().to_string(index=False))

In [ ]:
# Compare exact values versus rounded values
tmp = com_df.copy()
tmp["COM_round6"] = tmp["COM"].round(6)

print("Exact unique COM values:", tmp["COM"].nunique())
print("Rounded-to-6-decimals unique COM values:", tmp["COM_round6"].nunique())

print("\nRounded value counts:")
print(tmp["COM_round6"].value_counts().sort_index())

In [ ]:
# Cell K: Betti curve per architecture — mean ± std over 30 seeds
# One figure per (arch, activation) pair.  Individual seed traces shown faint.
# Matches the PCN graph_betti_numbers style from Trainer.py.

import numpy as _np

def plot_betti_curve_ann(
    arch_name:          str,
    act_name:           str,
    eta:                float = PLOT_ETA,
    dims:               tuple = DIMS,
    n_seeds:            int   = N_SEEDS,
    dir_name:           str   = DIR_NAME,
    use_running_min:    bool  = False,      # False = raw curves (like PCN default)
    color:              str   = 'blue',
    plot_individual:    bool  = True,
    alpha_individual:   float = 0.12,
    lw_individual:      float = 1.0,
    lw_mean:            float = 2.0,
    marker:             str   = 's',
    figsize:            tuple = (10, 6),
    title:              str   = None,
    save:               bool  = False,
    filename:           str   = None,
):
    """
    Plot the Betti curve (hidden layers only) for one (arch, activation) pair.
    Shows all individual seed traces faintly, overlaid with the mean line
    and a ±1 std band — matching the PCN graph_betti_numbers style.

    X-axis: layer index  (0 = first hidden layer, L-1 = last hidden layer).
    Y-axis: β_{dim} (or sum of dims) at the given eta threshold.
    """
    # ---- build betti matrix (K, L) -----------------------------------------
    betti_mat = get_betti_mat_v3(
        arch_name       = arch_name,
        act_name        = act_name,
        n_seeds         = n_seeds,
        eta             = eta,
        dims            = dims,
        dir_name        = dir_name,
    )                                          # (K, L)

    if use_running_min:
        betti_mat = _np.minimum.accumulate(betti_mat, axis=1)

    K, L = betti_mat.shape
    mean_curve = betti_mat.mean(axis=0)        # (L,)
    std_curve  = betti_mat.std(axis=0, ddof=1) # (L,)
    x          = _np.arange(L)

    # ---- layer labels -------------------------------------------------------
    # Hidden layers: label as 1 .. L
    layer_labels = [str(i + 1) for i in range(L)]

    # ---- plot ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    # Individual seed traces (faint)
    if plot_individual:
        for i in range(K):
            ax.plot(x, betti_mat[i], color=color,
                    alpha=alpha_individual, linewidth=lw_individual)

    # Mean line
    ax.plot(x, mean_curve, color=color, linewidth=lw_mean,
            marker=marker, label=f'Mean (n={K})')

    # ±1 std band
    ax.fill_between(x, mean_curve - std_curve, mean_curve + std_curve,
                    color=color, alpha=0.2, linewidth=0, label='±1 std')

    # Axes
    ax.set_xticks(x)
    ax.set_xticklabels(layer_labels, rotation=0)
    ax.set_xlabel('Hidden layer', fontsize=14, labelpad=6)

    betti_str = r'$\beta_{' + str(dims[0]) + r'}$'
    for d in dims[1:]:
        betti_str += r' + $\beta_{' + str(d) + r'}$'
    ax.set_ylabel(betti_str, fontsize=14, labelpad=6)

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.tick_params(axis='both', which='major', labelsize=11)

    # y-ticks: integer steps when range is small
    ymax = int(_np.max(betti_mat))
    if ymax <= 20:
        ax.set_yticks(_np.arange(0, ymax + 2, 1))

    if title is None:
        title = (
            rf'{betti_str} — {arch_name} / {act_name},  '
            rf'$\eta={eta}$,  K={K} seeds'
        )
    ax.set_title(title, fontsize=15, pad=10)

    plt.tight_layout()

    if save:
        dim_str  = ''.join(str(d) for d in dims)
        fname    = filename if filename else f'{arch_name}_{act_name}'
        out_path = FIG_ROOT / f'betti_curves/{fname}_B{dim_str}_eta{eta}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'  Saved → {out_path}')

    plt.show()
    plt.close(fig)

    return mean_curve, std_curve, betti_mat


# ── Run: one figure per (arch × activation) pair, saved automatically ────────
SAVE_BETTI_FIGS = True   # set False to only display, not save

for arch_name in ARCHS:
    for act_name in ACTIVATIONS:
        print(f'Plotting {arch_name} / {act_name} ...')
        plot_betti_curve_ann(
            arch_name       = arch_name,
            act_name        = act_name,
            eta             = PLOT_ETA,
            dims            = DIMS,
            n_seeds         = N_SEEDS,
            dir_name        = DIR_NAME,
            use_running_min = False,
            save            = SAVE_BETTI_FIGS,
        )
